In [ ]:
# Build the strict drug-gene table (Drugs_direct.xlsx) from ChEMBL mechanism-of-action records.
#
# Input : Drugs_1.xlsx  (curated drug list; columns Gene, Drug, ChEMBL_ID, Action, Boolean_Value)
# Output: ChEMBL_mechanisms_raw.csv  (all mechanism records retrieved, with the decision taken)
#         Drugs_direct.xlsx          (strict drug-gene associations used by the ASP optimization)
#
# Rule: a drug-gene pair is retained only if the ChEMBL target is a SINGLE PROTEIN, the
# interaction is annotated as direct, and the action type is clearly inhibitory (Boolean 0)
# or activating (Boolean 1). Run with ChEMBL release 37 (the script prints the release served).

import time
import requests
import pandas as pd

BASE = "https://www.ebi.ac.uk/chembl/api/data"
SESSION = requests.Session()


def get(url, params=None):
    for k in range(5):
        r = SESSION.get(url, params=params, timeout=60)
        if r.status_code == 200:
            return r.json()
        time.sleep(2 * (k + 1))
    r.raise_for_status()


print("ChEMBL version:", get(f"{BASE}/status.json").get("chembl_db_version"))

drugs = pd.read_excel("Drugs_1.xlsx", usecols=range(5)).dropna(how="all")
ids = sorted(drugs["ChEMBL_ID"].dropna().unique())
name_of = drugs.drop_duplicates("ChEMBL_ID").set_index("ChEMBL_ID")["Drug"].to_dict()

INHIBITORY = {"INHIBITOR", "ANTAGONIST", "BLOCKER", "NEGATIVE ALLOSTERIC MODULATOR",
              "NEGATIVE MODULATOR", "INVERSE AGONIST", "ANTISENSE INHIBITOR",
              "RNAI INHIBITOR", "GATING INHIBITOR"}
ACTIVATING = {"AGONIST", "PARTIAL AGONIST", "ACTIVATOR", "OPENER",
              "POSITIVE ALLOSTERIC MODULATOR", "POSITIVE MODULATOR"}

target_cache = {}


def target_info(target_id):
    if target_id not in target_cache:
        t = get(f"{BASE}/target/{target_id}.json")
        genes = sorted({s["component_synonym"]
                        for c in t.get("target_components", [])
                        for s in c.get("target_component_synonyms", [])
                        if s.get("syn_type") == "GENE_SYMBOL"})
        target_cache[target_id] = (t.get("target_type"), genes)
    return target_cache[target_id]


rows = []
for cid in ids:
    data = get(f"{BASE}/mechanism.json", {"molecule_chembl_id": cid, "limit": 1000})
    for m in data.get("mechanisms", []):
        tid = m.get("target_chembl_id")
        ttype, genes = target_info(tid) if tid else (None, [])
        refs = "; ".join(f'{r.get("ref_type")}:{r.get("ref_id")}'
                         for r in (m.get("mechanism_refs") or []))
        rows.append(dict(ChEMBL_ID=cid, Drug=name_of.get(cid),
                         action_type=m.get("action_type"),
                         mechanism=m.get("mechanism_of_action"),
                         target_chembl_id=tid, target_type=ttype,
                         Genes="; ".join(genes),
                         direct_interaction=m.get("direct_interaction"),
                         mechanism_refs=refs))
mech = pd.DataFrame(rows)


def decide(r):
    if r.target_type != "SINGLE PROTEIN":
        return f"excluded: target is not a single protein ({r.target_type})"
    if not r.Genes:
        return "excluded: no gene symbol"
    if r.direct_interaction not in (1, True):
        return "excluded: interaction not direct"
    if r.action_type in INHIBITORY:
        return "0"
    if r.action_type in ACTIVATING:
        return "1"
    return f"excluded: action type not directional ({r.action_type})"


mech["decision"] = mech.apply(decide, axis=1)   # "0"/"1" = retained with Boolean state
mech.to_csv("ChEMBL_mechanisms_raw.csv", index=False)

kept = mech[mech.decision.isin(["0", "1"])].copy()
kept["Boolean_Value"] = kept.decision.astype(int)
kept["Action"] = kept.Boolean_Value.map({0: "Inhibition", 1: "Activation"})
kept["Gene"] = kept.Genes.str.split("; ")
kept = kept.explode("Gene")
direct = kept[["Gene", "Drug", "ChEMBL_ID", "Action", "Boolean_Value",
               "mechanism", "target_chembl_id", "mechanism_refs"]].drop_duplicates()
direct.to_excel("Drugs_direct.xlsx", index=False)

print("Retained drug-gene rows:", len(direct), "| drugs:", direct.Drug.nunique(),
      "| genes:", direct.Gene.nunique())
print(mech.decision.str.split(" \\(").str[0].value_counts())
